In [1]:
from datetime import datetime
# TENSORFLOW AND KERAS THINGS
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers
import numpy as np
import pandas as pd
import tensorflow as tf
from dqn_utils.utils import create_q_model

# ENV THINGS
from adabyron_environment.mercadoSecundario_v2 import MercadoSecundario
from adabyron_environment.agents.storageBattery_mercado_secundario import StorageBatteryMS
import matplotlib.pyplot as plt
import random

2024-07-19 07:44:16.103353: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-07-19 07:44:16.144299: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-19 07:44:16.144332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-19 07:44:16.145578: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-19 07:44:16.152547: I tensorflow/core/platform/cpu_feature_guar

In [2]:
gpus = tf.config.list_physical_devices('GPU')
print(gpus)
tf.config.set_visible_devices(gpus[1], 'GPU')
tf.config.experimental.set_memory_growth(gpus[1], True)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
SEED = 42
GAMMA = 0.99 # Discount factor for past rewards
EPSILON = 1.0 # Epsilon greedy parameter
EPSILON_MIN = 0.1 # Minimum epsilon greedy parameter
EPSILON_MAX = 1.0 # Decay rate for epsilon greedy parameter

epsilon_interval = (
    EPSILON_MAX - EPSILON_MIN
)  # Rate at which to reduce chance of random action being taken

BATCH_SIZE = 32 # Size of batch taken from replay buffer
MAX_STEPS_PER_EPISODE = 100000
MAX_EPISODES = 100000^10 # Number of episodes to run

In [4]:
#################################################
######### ENVIRONMENT INITIALIZATION ############
#################################################
bolsa_meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

bandaSecundaria_df = pd.read_csv('test_data/bandaSecundaria.csv', sep=';')
regulacionSubir_df = pd.read_csv('test_data/regulacionSubir.csv', sep=';')
regulacionBajar_df = pd.read_csv('test_data/regulacionBajar.csv', sep=';')
porcentajeSubir_df = pd.read_csv('test_data/porcentajeSubir.csv', sep=';')
porcentajeBajar_df = pd.read_csv('test_data/porcentajeBajar.csv', sep=';')


bandaSecundaria = []
regulacionSubir = []
regulacionBajar = []
porcentajeSubir = []
porcentajeBajar = []

energy_fixed_input = []


def getNewEnvironment(bolsa_meses, bandaSecundaria_df, regulacionSubir_df, regulacionBajar_df, porcentajeSubir_df, porcentajeBajar_df):
    if not bolsa_meses:
        bolsa_meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
        
    month = random.choice(bolsa_meses)
    bolsa_meses.remove(month)
    
    bandaSecundaria = bandaSecundaria_df[month]
    regulacionSubir = regulacionSubir_df[month]
    regulacionBajar = regulacionBajar_df[month]
    porcentajeSubir = porcentajeSubir_df[month]
    porcentajeBajar = porcentajeBajar_df[month]
    
    energy_fixed_input = np.array([bandaSecundaria, regulacionSubir, porcentajeSubir, regulacionBajar, porcentajeBajar]).flatten()

    
        
    battery_capacity = random.randint(30, 70)
    battery = StorageBatteryMS("B1",capacity= battery_capacity, original_soc = random.randint(0, battery_capacity))
    mercadoSecundario = MercadoSecundario(preservedEnergy=0.50, 
                                      priceCurve= bandaSecundaria, 
                                      highCurve= regulacionSubir, 
                                      lowCurve= regulacionBajar,
                                      highPercentage= porcentajeSubir,
                                      lowPercentage= porcentajeBajar,
                                      battery= battery)
        
    return bandaSecundaria, regulacionSubir, regulacionBajar, porcentajeSubir, porcentajeBajar, energy_fixed_input, bolsa_meses, battery, mercadoSecundario

########## REDUCE DAY LENGTH FOR EARLY TESTING PURPOSES
#bandaSecundaria = np.array(bandaSecundaria).reshape(-1,3).mean(axis=1)
#regulacionSubir = np.array(regulacionSubir).reshape(-1,3).mean(axis=1)
#regulacionBajar = np.array(regulacionBajar).reshape(-1,3).mean(axis=1)
#porcentajeSubir = np.array(porcentajeSubir).reshape(-1,3).mean(axis=1)
#porcentajeBajar = np.array(porcentajeBajar).reshape(-1,3).mean(axis=1)

battery = None
mercadoSecundario = None

bandaSecundaria, regulacionSubir, regulacionBajar, porcentajeSubir, porcentajeBajar, energy_fixed_input, bolsa_meses, battery, mercadoSecundario = getNewEnvironment(bolsa_meses, bandaSecundaria_df, regulacionSubir_df, regulacionBajar_df, porcentajeSubir_df, porcentajeBajar_df)

In [5]:
#################################################
######### DQN MODEL INITIALIZATION ##############
#################################################

if battery is not None and mercadoSecundario is not None:
    print("Battery and Market initialized")
else:
    raise Exception("Battery and Market not initialized")

model = create_q_model(battery.get_action_space())
model_target = create_q_model(battery.get_action_space())

model.load_weights("models/Dueling/model_episode_99914.h5")
model_target.load_weights("models/Dueling/model_target_episode_99914.h5")

optimizer = keras.optimizers.Adam(learning_rate=0.00025, clipnorm=1.0)

# Experience replay buffers
action_history = []
state_history = []
state_next_history = []
rewards_history = []
done_history = []
episode_reward_history = []
running_reward = 0
episode_count = 0
frame_count = 0
medias = []
# Number of frames to take random action and observe output
epsilon_random_frames = 50000
# Number of frames for exploration
epsilon_greedy_frames = 1000000.0
# Maximum replay length
max_memory_length = 100000

# Train the model after 4 actions
update_after_actions = 4
# How often to update the target network
update_target_network = 10000
# Using huber loss for stability
loss_function = keras.losses.Huber()

save_model_path = "models/"

Battery and Market initialized


2024-07-19 07:44:18.420170: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 31141 MB memory:  -> device: 1, name: Tesla V100-PCIE-32GB, pci bus id: 0000:d7:00.0, compute capability: 7.0


In [6]:
def dibujaGraficaQ():
    print("Dibujo graficas")
    
    
    
    bandaSecundaria = bandaSecundaria_df["enero"]
    regulacionSubir = regulacionSubir_df["enero"]
    regulacionBajar = regulacionBajar_df["enero"]
    porcentajeSubir = porcentajeSubir_df["enero"]
    porcentajeBajar = porcentajeBajar_df["enero"]
    
    energy_fixed_input = np.array([bandaSecundaria, regulacionSubir, porcentajeSubir, regulacionBajar, porcentajeBajar]).flatten()
    
    test_battery = StorageBatteryMS("B1",capacity= 36, original_soc = 23)
    test_mercadoSecundario = MercadoSecundario(preservedEnergy=0.50, 
                                        priceCurve= bandaSecundaria, 
                                        highCurve= regulacionSubir, 
                                        lowCurve= regulacionBajar,
                                        highPercentage= porcentajeSubir,
                                        lowPercentage= porcentajeBajar,
                                        battery= test_battery)
    
    proto_testing_state = test_mercadoSecundario.reset()
    
    testing_qvalues = []
    while True:
        print(proto_testing_state.day_step)
        testing_state = [proto_testing_state.day_step, 
                         proto_testing_state.cumulative_consumption, 
                         proto_testing_state.cumulative_price, 
                         proto_testing_state.agents_info['B1']['soc'], 
                         test_battery.capacity, 
                         proto_testing_state.agents_info['B1']['chargeHistory'], 
                         proto_testing_state.agents_info['B1']['dischargeHistory']] + energy_fixed_input.tolist()
        
        testing_state_tensor = tf.convert_to_tensor([testing_state])
        testing_action_probs = model(testing_state_tensor, training=False)
        testing_action = tf.argmax(testing_action_probs[0]).numpy()
        
        testing_qvalues.append(tf.reduce_max(testing_action_probs).numpy())
        
        proto_testing_state_next, _, test_done = test_mercadoSecundario.step_dqn(proto_testing_state, testing_action)
        proto_testing_state = proto_testing_state_next
        
        if test_done: break
    medias.append(tf.reduce_mean(testing_qvalues).numpy())
    
    plt.plot(medias)
    plt.savefig("images/DQN/qvalues_{}.jpg".format(datetime.now().strftime('%H_%M_%S')))
    plt.close()

In [ ]:
#################################################
######### DQN TRAINING ##########################
#################################################
while True:
    bandaSecundaria, regulacionSubir, regulacionBajar, porcentajeSubir, porcentajeBajar, energy_fixed_input, bolsa_meses, battery, mercadoSecundario = getNewEnvironment(bolsa_meses, bandaSecundaria_df, regulacionSubir_df, regulacionBajar_df, porcentajeSubir_df, porcentajeBajar_df)
    
    # Get first state
    proto_state = mercadoSecundario.reset()
    state = [proto_state.day_step, proto_state.cumulative_consumption, proto_state.cumulative_price, proto_state.agents_info['B1']['soc'], battery.capacity, proto_state.agents_info['B1']['chargeHistory'], proto_state.agents_info['B1']['dischargeHistory']] + energy_fixed_input.tolist()
    episode_reward = 0
    
    for timestep in range(1, MAX_STEPS_PER_EPISODE):
        frame_count += 1
        
        # Use epsilon-greedy for exploration
        #if frame_count < epsilon_random_frames or EPSILON > np.random.rand(1)[0]:
        #    # Take random action
        #    action = np.random.choice(battery.get_action_space())
        #else:
        # Predict action Q-values
        # From environment state
        state_tensor = tf.convert_to_tensor([state])
        action_probs = model(state_tensor, training=False)
        # Take best action
        action = tf.argmax(action_probs[0]).numpy()
        
        # Decay probability of taking random action
        EPSILON -= epsilon_interval / epsilon_greedy_frames
        EPSILON = max(EPSILON, EPSILON_MIN)
        
        # Apply the sampled action in our environment
        proto_state_next, reward, done = mercadoSecundario.step_dqn(proto_state, action)
        state_next = [proto_state_next.day_step, proto_state_next.cumulative_consumption, proto_state_next.cumulative_price, proto_state_next.agents_info['B1']['soc'], battery.capacity, proto_state_next.agents_info['B1']['chargeHistory'], proto_state_next.agents_info['B1']['dischargeHistory']] + energy_fixed_input.tolist()

        episode_reward += reward
        
        # Save actions and states in replay buffer
        action_history.append(action)
        state_history.append(state)
        state_next_history.append(state_next)
        done_history.append(done)
        rewards_history.append(reward)
        state       = state_next
        proto_state = proto_state_next
        # Update weights of the model
        if frame_count % update_after_actions == 0 and len(done_history) > BATCH_SIZE:
            #print("Training model")
            # Get indices of samples for replay buffers
            indices = np.random.choice(range(len(done_history)), size=BATCH_SIZE)
            
            # Using list comprehension to sample from replay buffer
            state_sample = np.array([state_history[i] for i in indices])
            state_next_sample = np.array([state_next_history[i] for i in indices])
            rewards_sample = [rewards_history[i] for i in indices]
            action_sample = [action_history[i] for i in indices]
            done_sample = tf.convert_to_tensor(
                [float(done_history[i]) for i in indices]
            )
            
            # Build the updated Q-values for the sampled future states
            # Use the target model for stability
            future_rewards = model_target.predict(state_next_sample, verbose=0)
            # Q value = reward + discount factor * expected future reward
            updated_q_values = rewards_sample + GAMMA * tf.reduce_max(
                future_rewards, axis=1
            )
            
            # If final frame set the last value to -1
            updated_q_values = updated_q_values * (1 - done_sample) - done_sample
            
            # Create a mask so we only calculate loss on the updated Q-values
            masks = tf.one_hot(action_sample, battery.get_action_space())
            
            with tf.GradientTape() as tape:
                # Train the model on the states and updated Q-values
                q_values = model(state_sample)
                
                # Apply the masks to the Q-values to get the Q-value for action taken
                q_action = tf.reduce_sum(tf.multiply(q_values, masks), axis=1)
                # Calculate loss between new Q-value and old Q-value
                loss = loss_function(updated_q_values, q_action)
            
            # Backpropagation
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables), verbose=0)


        if frame_count % update_target_network == 0:
            print("Updating target model")
            # update the target network with new weights
            model_target.set_weights(model.get_weights())
            # Log details
            template = "running reward: {:.2f} at episode {}, frame count {}"
            print(template.format(running_reward, episode_count, frame_count))
            model.save_weights(save_model_path + "model_episode_{}.h5".format(episode_count))
            model_target.save_weights(save_model_path + "model_target_episode_{}.h5".format(episode_count))
            dibujaGraficaQ()
            
        # Limit the state and reward history
        if len(rewards_history) > max_memory_length:
            del rewards_history[:1]
            del state_history[:1]
            del state_next_history[:1]
            del action_history[:1]
            del done_history[:1]
            
        if done:
            break
        
    # Update running reward to check condition for solving
    episode_reward_history.append(episode_reward)
    if len(episode_reward_history) > 100:
        del episode_reward_history[:1]
    running_reward = np.mean(episode_reward_history)
    
    episode_count += 1
    
    #if running_reward > 40:  # Condition to consider the task solved
    #    print("Solved at episode {}!".format(episode_count))
    #    break
    
    #if (
    #    MAX_EPISODES > 0 and episode_count >= MAX_EPISODES
    #):
    #    print("Stopped at episode {}!".format(episode_count))
    #    break

print(max(rewards_history))

2024-07-19 07:44:21.001709: I external/local_xla/xla/service/service.cc:168] XLA service 0x560797b73320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-07-19 07:44:21.001745: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): Tesla V100-PCIE-32GB, Compute Capability 7.0
2024-07-19 07:44:21.006576: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-07-19 07:44:21.023858: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8906
I0000 00:00:1721375061.165972     672 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Updating target model
running reward: 23.95 at episode 416, frame count 10000
Dibujo graficas
-1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
Updating target model
running reward: 23.78 at episode 833, frame count 20000
Dibujo graficas
-1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
